# TogoLM Fine-Tuning — Kaggle T4 (Unsloth)

Fine-tune Mistral 7B Instruct v0.3 on the TogoLM corpus using QLoRA via Unsloth.

**Setup checklist before running:**
- [ ] Accelerator set to GPU T4 x2 (Session options)
- [ ] Internet ON
- [ ] Dataset `togolm-sft` added (contains train + eval JSONL)
- [ ] Secret `HF_TOKEN` added with your HuggingFace token
- [ ] Mistral-7B-Instruct-v0.3 license accepted on HuggingFace

In [ ]:
# 1. Patch bitsandbytes + install deps
import os

# Patch the two files that import triton.ops (non-existent in newer triton).
# We replace them with stubs that export the expected symbols as None —
# these triton int8 kernels are NOT used in our QLoRA 4-bit NF4 pipeline.
PATCHES = {
    "/usr/local/lib/python3.12/dist-packages/bitsandbytes/triton/int8_matmul_mixed_dequantize.py": (
        "int8_matmul_mixed_dequantize = None\n"
    ),
    "/usr/local/lib/python3.12/dist-packages/bitsandbytes/triton/dequantize_rowwise.py": (
        "dequantize_rowwise = None\n"
    ),
}

for path, content in PATCHES.items():
    if os.path.exists(path):
        with open(path, "w") as f:
            f.write(content)
        print(f"Patched: {path}")

# Install unsloth and dependencies
!pip install -q unsloth "transformers==4.45.2" datasets huggingface_hub

print("Done — run all remaining cells in order.")

In [ ]:
# 2. Importer unsloth EN PREMIER (avant transformers/peft/bitsandbytes)
import unsloth  # noqa: F401 — must be first import

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)
print("HuggingFace login OK")

In [ ]:
# 3. Check GPU
import torch

for i in range(torch.cuda.device_count()):
    name = torch.cuda.get_device_name(i)
    vram = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"GPU {i}: {name} — {vram:.1f} GB")

In [ ]:
# 4. Locate dataset files
import glob

train_files = glob.glob("/kaggle/input/**/togolm_sft_train.jsonl", recursive=True)
eval_files  = glob.glob("/kaggle/input/**/togolm_sft_eval.jsonl",  recursive=True)

assert train_files, "togolm_sft_train.jsonl not found — did you add the dataset?"
assert eval_files,  "togolm_sft_eval.jsonl not found"

TRAIN_FILE = train_files[0]
EVAL_FILE  = eval_files[0]
print(f"Train : {TRAIN_FILE}")
print(f"Eval  : {EVAL_FILE}")

In [ ]:
# 5. Configuration
BASE_MODEL  = "mistralai/Mistral-7B-Instruct-v0.3"
OUTPUT_DIR  = "/kaggle/working/togolm-7b"
HF_REPO     = "togolm/togolm-7b-instruct-v1"
CORPUS_REPO = "togolm/togolm-corpus-v1"

LORA_R      = 16
LORA_ALPHA  = 32
EPOCHS      = 3
BATCH_SIZE  = 2
GRAD_ACCUM  = 8    # effective batch = 16
LR          = 2e-4
MAX_SEQ_LEN = 2048

In [ ]:
# 6. Load model with Unsloth (4-bit QLoRA, handles CUDA deps automatically)
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    token=hf_token,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
model.print_trainable_parameters()

In [ ]:
# 7. Load dataset
import json
from datasets import Dataset, DatasetDict

def read_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

ALPACA_TEMPLATE = """Below is an instruction about Togo. Write a response that answers it accurately.

### Instruction:
{instruction}

### Response:
{output}"""

def formatting_fn(example):
    return [ALPACA_TEMPLATE.format(
        instruction=ex["instruction"],
        output=ex["output"],
    ) for ex in example]

dataset = DatasetDict({
    "train": Dataset.from_list(read_jsonl(TRAIN_FILE)),
    "eval":  Dataset.from_list(read_jsonl(EVAL_FILE)),
})
print(f"Train: {len(dataset['train'])} | Eval: {len(dataset['eval'])}")

In [ ]:
# 8. Train
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    bf16=False,
    fp16=True,          # T4 ne supporte pas bf16
    logging_steps=10,
    save_steps=100,
    eval_steps=100,
    eval_strategy="steps",
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["eval"],
    formatting_func=formatting_fn,
    max_seq_length=MAX_SEQ_LEN,
    tokenizer=tokenizer,
    args=training_args,
)

trainer.train()

In [ ]:
# 9. Save adapter
trainer.save_model(f"{OUTPUT_DIR}/final")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final")
print(f"Adapter saved to {OUTPUT_DIR}/final")

In [ ]:
# 10. Push to HuggingFace Hub
if HF_REPO:
    model.push_to_hub(HF_REPO, token=hf_token)
    tokenizer.push_to_hub(HF_REPO, token=hf_token)
    print(f"Pushed to https://huggingface.co/{HF_REPO}")
else:
    print("HF_REPO non rempli — télécharge l'adapter depuis /kaggle/working/togolm-7b/final")

In [ ]:
# 10b. Publier le dataset SFT sur togolm/togolm-corpus-v1
from huggingface_hub import HfApi

api = HfApi(token=hf_token)

for local_path, repo_path in [
    (TRAIN_FILE, "sft/togolm_sft_train.jsonl"),
    (EVAL_FILE,  "sft/togolm_sft_eval.jsonl"),
]:
    api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo=repo_path,
        repo_id=CORPUS_REPO,
        repo_type="dataset",
    )
    print(f"Uploaded {repo_path} -> huggingface.co/datasets/{CORPUS_REPO}")

In [ ]:
# 11. Test rapide
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

question = "Quel est le taux d'imposition sur les sociétés au Togo ?"
prompt = f"### Instruction:\n{question}\n\n### Response:\n"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=256, do_sample=False)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))